In [2]:
# Load dataset
import pandas as pd

df = pd.read_csv("ecommerce_sales_analytics_5000.csv", parse_dates=["order_date"])

In [3]:
# Verify
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          5000 non-null   int64         
 1   order_date        5000 non-null   datetime64[ns]
 2   customer_id       5000 non-null   int64         
 3   product_category  5000 non-null   object        
 4   region            5000 non-null   object        
 5   quantity          5000 non-null   int64         
 6   unit_price        5000 non-null   float64       
 7   discount          5000 non-null   float64       
 8   payment_method    5000 non-null   object        
 9   delivery_days     5000 non-null   int64         
 10  customer_rating   5000 non-null   float64       
 11  revenue           5000 non-null   float64       
dtypes: datetime64[ns](1), float64(4), int64(4), object(3)
memory usage: 468.9+ KB


In [6]:
# map() - Label high-value orders
df["payment_type"] = df["payment_method"].map({
    "Card": "Digital",
    "Wallet": "Digital",
    "COD": "Cash"
})
# Explanation: Each value is mapped individually -> fast & simple.

In [7]:
# Verify
df["payment_type"].unique()       #Check unique values

array(['Digital', 'Cash'], dtype=object)

In [8]:
df[["payment_method", "payment_type"]].head()      #Compare before vs after

,payment_method,payment_type
0,Wallet,Digital
1,Card,Digital
2,COD,Cash
3,Wallet,Digital
4,Wallet,Digital


In [9]:
df.groupby("payment_type").size()       #Business sanity check

payment_type
Cash       1774
Digital    3226
dtype: int64

In [11]:
# apply() - Create revenue category
def revenue_category(row):
    if row["revenue"] > 500:
        return "High"
    elif row["revenue"] >= 200:
        return "Medium"
    
df["revenue_category"] = df.apply(revenue_category,axis=1)

# why apply? Decision depend on multiple coloumn

In [13]:
# Verify 
df["revenue_category"].value_counts()  # Checking Category

revenue_category
High      3252
Medium    1098
Name: count, dtype: int64

In [14]:
df[["revenue", "revenue_category"]].sample(10)          #Validate logic manually
# Revenue > 500 -> High
# Between 200-500 -> Medium
# < 200 -> Low

,revenue,revenue_category
2927,111.49,None
4298,798.57,High
4901,2097.19,High
2134,1711.41,High
4924,57.71,None
3497,1024.35,High
1049,573.12,High
3886,1319.42,High
1782,618.95,High
1467,350.19,Medium


In [15]:
# pipe() - Method chaining
def add_profit(data):
    data["profit"] = data["revenue"] * 0.20
    return data

def add_net_revenue(data):
    data["net_revenue"] = data["revenue"] - data["profit"]
    return data

df = (
    df
    .pipe(add_profit)
    .pipe(add_net_revenue)
)

# Verify
df[["revenue", "profit", "net_revenue"]].head()          #Check new columns

,revenue,profit,net_revenue
0,1883.20,376.640,1506.560
1,304.10,60.820,243.280
2,644.35,128.870,515.480
3,2569.90,513.980,2055.920
4,468.56,93.712,374.848
